# STAT 764 · Meeting 6 — Comparing models honestly

**Thursday, September 24**

Tuesday you picked a model by looking at a number. Today: what has to be true for that
number to support the comparison you used it for.

Cross-validation fixes the noise problem from Meeting 3. It does **not** fix
everything, and the assumption it quietly makes is the subject of this meeting.

In [ ]:
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

import numpy as np
import pandas as pd
from stat764 import load

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 1. The assumption inside cross-validation

`KFold(shuffle=True)` puts rows into folds at random. That is a claim about your data:
**that the rows are interchangeable** — that a row's position carries no information
about what it looks like.

For a pile of house sales, plausible. For anything measured over time, not obviously.

Nothing in the code will tell you which situation you are in. You have to check.

## 2. Ames: check, and find nothing

The Ames data spans 2006–2010. An assessor in 2010 does not get to train on 2010 — they
have the past and must predict the present. So evaluate it that way, and compare with
what random cross-validation claimed.

In [ ]:
ames = load("ames.csv")
NUM = ["Gr_Liv_Area", "Year_Built", "Overall_Qual", "Total_Bsmt_SF", "Lot_Area"]
CAT = ["Neighborhood", "Central_Air"]


def ames_pipe(model=None):
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                              ("scale", StandardScaler())]), NUM),
            ("cat", Pipeline([("fill", SimpleImputer(strategy="constant",
                                                     fill_value="Missing")),
                              ("encode", OneHotEncoder(handle_unknown="ignore"))]), CAT),
        ])),
        ("model", model or LinearRegression()),
    ])


X, y = ames[NUM + CAT], ames["SalePrice"]

shuffled = cross_val_score(ames_pipe(), X, y,
                           cv=KFold(5, shuffle=True, random_state=0), scoring="r2")

past, present = ames[ames["Yr_Sold"] <= 2009], ames[ames["Yr_Sold"] == 2010]
assessor = ames_pipe().fit(past[NUM + CAT], past["SalePrice"])
assessor_r2 = r2_score(present["SalePrice"], assessor.predict(present[NUM + CAT]))

print(f"  random 5-fold CV                R-squared {shuffled.mean():.3f}")
print(f"  train 2006-2009, predict 2010   R-squared {assessor_r2:.3f}")
print(f"  random CV optimistic by         {shuffled.mean() - assessor_r2:+.3f}")

**Nothing.** The honest evaluation is, if anything, slightly *better* than the one that
shuffled across years.

That is a real result and it is worth pausing on, because it is not what you were
braced for. Look at why:

In [ ]:
print("  median sale price by year")
print(ames.groupby("Yr_Sold")["SalePrice"].median().to_string())
print(f"\n  2006 -> 2010: "
      f"{100 * (ames.groupby('Yr_Sold')['SalePrice'].median().loc[2010] / ames.groupby('Yr_Sold')['SalePrice'].median().loc[2006] - 1):+.1f}%")

Ames barely moved. The relationship between a house's features and its price was
essentially the same in 2010 as in 2006, so shuffling across years costs nothing.

⚠ **You could not have known that without checking**, and "I checked and it was fine"
is a finding. It is the sentence that makes the next model's claim credible.

## 3. Bike sharing: check, and find plenty

Same check, different data. Hourly bike rentals across 2011–2012.

In [ ]:
bike = load("bike_hour.csv")
bike["dteday"] = pd.to_datetime(bike["dteday"])
FEAT = [c for c in bike.columns
        if c not in ("cnt", "casual", "registered", "dteday", "instant")]


def bike_pipe(model):
    return Pipeline([("fill", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler()),
                     ("model", model)])


Xb, yb = bike[FEAT], bike["cnt"]
cut = bike["dteday"].quantile(0.75)
before, after = bike[bike["dteday"] <= cut], bike[bike["dteday"] > cut]

rows = []
for name, model in [("OLS", LinearRegression()),
                    ("RandomForest", RandomForestRegressor(n_estimators=60,
                                                           random_state=0, n_jobs=-1))]:
    shuf = cross_val_score(bike_pipe(model), Xb, yb,
                           cv=KFold(5, shuffle=True, random_state=0), scoring="r2").mean()
    fit = bike_pipe(model).fit(before[FEAT], before["cnt"])
    fwd = r2_score(after["cnt"], fit.predict(after[FEAT]))
    rows.append({"model": name, "random CV": shuf, "train past -> predict future": fwd,
                 "optimism": shuf - fwd})

print(pd.DataFrame(rows).round(3).to_string(index=False))

Both models look materially worse once they have to predict forwards. And the reason
is not subtle:

In [ ]:
bike["month"] = bike["dteday"].dt.to_period("M")
monthly = bike.groupby("month")["cnt"].mean()
print(f"  mean hourly rentals, first 6 months : {monthly.head(6).mean():.0f}")
print(f"  mean hourly rentals, last 6 months  : {monthly.tail(6).mean():.0f}")
print(f"  change                              : "
      f"{100 * (monthly.tail(6).mean() / monthly.head(6).mean() - 1):+.0f}%")

Ridership roughly **doubled**. A shuffled fold gets to train on late-2012 hours and
predict early-2011 ones — it has seen the future. The forward split cannot.

## 4. So which is right?

Both numbers are correct answers to different questions:

| | answers |
|---|---|
| **Random CV** | how well does this model do on *more data like the data I have*? |
| **Forward split** | how well will this model do *next month*? |

If you are deploying a model to predict next month, the second is the one you are
entitled to quote — and on the bike data the first overstates it by a wide margin.

⚠ **The point is not "always split by time."** On Ames that would have thrown away
information for no gain. The point is that **random CV makes an assumption you can
test in about four lines, and you do not get to skip testing it.**

## Studio

1. Reproduce the bike comparison with a third model of your choice. Does the size of
   the optimism depend on the model?
2. Bike has a `yr` column (0 = 2011, 1 = 2012). Drop it and repeat. Does the forward
   number get better or worse, and why?
3. Build a forward-split evaluation for Ames using **month** rather than year — 2010
   only, first nine months to predict the rest. Does anything appear at that resolution?

In [ ]:
# YOUR CODE HERE

## Compare

1. Which model had the largest gap between shuffled and forward? Why that one?
2. Question 2 — did dropping `yr` help or hurt, and what does your answer imply about
   what the model was using it for?
3. If you had only ever run random CV on the bike data, what would you have promised?

## Exit ticket

> Name a dataset from your own field where shuffling rows would be dishonest, and say
> what the honest split would be.

## Also in this neighborhood

📗 **`TimeSeriesSplit`.** scikit-learn's forward-chaining cross-validator: several
forward splits rather than one, so you get a spread instead of a single number.

🎓 **Distribution shift** as a field — detecting it, and what to do when you find it.
That is Meeting 26.

🚫 **Shuffled CV on grouped data.** The same failure with a different index: Meeting 9
has repeat patients, where a random fold puts the same person on both sides.